In [6]:
import pandas as pd
import numpy as np

In [7]:
import sys
from pathlib import Path

sys.path.append(
    str(Path.cwd().parent)
)

In [35]:
import joblib
from pathlib import Path

models_dir = Path("../models")

In [36]:
logistic_model = joblib.load(
    models_dir / "pd_model.joblib"
)

In [37]:
lgd_model = joblib.load(
    models_dir / "lgd_model.joblib"
)

In [38]:
ead_model = joblib.load(
    models_dir / "ead_model.joblib"
)

In [8]:
from src.models.expected_loss import (
    calculate_expected_loss
)

In [39]:
df = pd.read_csv(
    "../data/raw/credit_portfolio.csv"
)

df["observation_date"] = pd.to_datetime(
    df["observation_date"]
)

In [43]:
oot = df[
    df["observation_date"] >= "2023-10-01"
].copy()

In [44]:
oot.shape

(30000, 19)

In [45]:
pd_features = [
    "age",
    "annual_income",
    "employment_years",
    "credit_score",
    "debt_to_income",
    "credit_utilization",
    "previous_defaults",
    "delinquencies_12m",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "unemployment_rate",
    "gdp_growth"
]

In [46]:
X_oot_pd = oot[pd_features]

In [48]:
oot_pd = logistic_model.predict_proba(
    X_oot_pd
)[:, 1]

In [49]:
pd.Series(oot_pd).describe()

count    30000.000000
mean         0.440317
std          0.165849
min          0.057679
25%          0.312843
50%          0.429412
75%          0.558063
max          0.959043
dtype: float64

In [50]:
lgd_features = [
    "credit_score",
    "debt_to_income",
    "credit_utilization",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "previous_defaults",
    "delinquencies_12m"
]

In [55]:
X_oot_lgd = oot[
    lgd_features
]

In [56]:
oot_lgd = lgd_model.predict(
    X_oot_lgd
)

In [59]:
oot_lgd = np.clip(
    oot_lgd,
    0,
    1
)

In [58]:
pd.Series(oot_lgd).describe()

count    30000.000000
mean         0.539100
std          0.033185
min          0.461851
25%          0.515120
50%          0.539059
75%          0.560334
max          0.651473
dtype: float64

In [60]:
print(
    "Minimum LGD:",
    oot_lgd.min()
)

print(
    "Maximum LGD:",
    oot_lgd.max()
)

print(
    "Average LGD:",
    oot_lgd.mean()
)

Minimum LGD: 0.46185139227131783
Maximum LGD: 0.6514733658058541
Average LGD: 0.5391002042233592


In [61]:
ead_features = [
    "age",
    "annual_income",
    "credit_score",
    "debt_to_income",
    "credit_utilization",
    "loan_amount",
    "loan_term_months",
    "interest_rate",
    "previous_defaults",
    "delinquencies_12m"
]

In [62]:
X_oot_ead = oot[
    ead_features
]

In [63]:
oot_ead = ead_model.predict(
    X_oot_ead
)

In [64]:
oot_ead = np.maximum(
    oot_ead,
    0
)

In [65]:
pd.Series(oot_ead).describe()

count    30000.000000
mean     11633.102793
std       7550.249704
min       1918.376800
25%       6486.856733
50%       9709.189128
75%      14537.797219
max      67379.069973
dtype: float64

In [67]:
risk_predictions = pd.DataFrame({
    "customer_id":
        oot["customer_id"].values,

    "observation_date":
        oot["observation_date"].values,

    "pd":
        oot_pd,

    "lgd":
        oot_lgd,

    "ead":
        oot_ead
})

In [68]:
risk_predictions.head()

,customer_id,observation_date,pd,lgd,ead
0,1,2023-10-01,0.413057,0.540942,8373.181314
1,1,2023-11-01,0.358039,0.528693,17731.262397
2,1,2023-12-01,0.701948,0.591822,3952.351414
3,2,2023-10-01,0.264907,0.528447,37763.597981
4,2,2023-11-01,0.311464,0.540262,3407.125559


In [69]:
risk_predictions["expected_loss"] = (
    calculate_expected_loss(
        risk_predictions["pd"],
        risk_predictions["lgd"],
        risk_predictions["ead"]
    )
)

In [70]:
risk_predictions.head()

,customer_id,observation_date,pd,lgd,ead,expected_loss
0,1,2023-10-01,0.413057,0.540942,8373.181314,1870.900842
1,1,2023-11-01,0.358039,0.528693,17731.262397,3356.398105
2,1,2023-12-01,0.701948,0.591822,3952.351414,1641.919103
3,2,2023-10-01,0.264907,0.528447,37763.597981,5286.495716
4,2,2023-11-01,0.311464,0.540262,3407.125559,573.324252


In [71]:
first_customer = risk_predictions.iloc[0]

print(
    "PD:",
    first_customer["pd"]
)

print(
    "LGD:",
    first_customer["lgd"]
)

print(
    "EAD:",
    first_customer["ead"]
)

print(
    "Expected Loss:",
    first_customer["expected_loss"]
)

PD: 0.41305654252304946
LGD: 0.5409420835292721
EAD: 8373.181313528012
Expected Loss: 1870.9008421462736


In [72]:
manual_el = (
    first_customer["pd"]
    * first_customer["lgd"]
    * first_customer["ead"]
)

print(
    "Manual Expected Loss:",
    manual_el
)

Manual Expected Loss: 1870.9008421462736


In [73]:
total_expected_loss = (
    risk_predictions[
        "expected_loss"
    ].sum()
)

In [77]:
print(
    f"Total Portfolio Expected Loss: "
    f"{total_expected_loss:,.2f}"
)

Total Portfolio Expected Loss: 83,644,576.09


In [75]:
total_ead = (
    risk_predictions["ead"]
    .sum()
)

print(
    f"Total EAD: "
    f"{total_ead:,.2f}"
)

Total EAD: 348,993,083.80


In [78]:
expected_loss_rate = (
    total_expected_loss
    / total_ead
)
print(
    f"Expected Loss / EAD: "
    f"{expected_loss_rate:.2%}"
)

Expected Loss / EAD: 23.97%


In [79]:
portfolio_pd = (
    risk_predictions["pd"]
    .mean()
)

In [80]:
portfolio_pd = (
    risk_predictions["pd"]
    .mean()
)

In [81]:
print(
    f"Portfolio Average PD: "
    f"{portfolio_pd:.2%}"
)

Portfolio Average PD: 44.03%


In [83]:
portfolio_lgd = (
    risk_predictions["lgd"]
    .mean()
)
print(
    f"Portfolio Average LGD: "
    f"{portfolio_lgd:.2%}"
)

Portfolio Average LGD: 53.91%


In [84]:
portfolio_summary = pd.DataFrame({
    "Metric": [
        "Number of Exposures",
        "Average PD",
        "Average LGD",
        "Total EAD",
        "Total Expected Loss",
        "Expected Loss / EAD"
    ],
    "Value": [
        len(risk_predictions),
        portfolio_pd,
        portfolio_lgd,
        total_ead,
        total_expected_loss,
        expected_loss_rate
    ]
})

In [85]:
portfolio_summary

,Metric,Value
0,Number of Exposures,3.000000e+04
1,Average PD,4.403169e-01
2,Average LGD,5.391002e-01
3,Total EAD,3.489931e+08
4,Total Expected Loss,8.364458e+07
5,Expected Loss / EAD,2.396740e-01


In [86]:
risk_predictions.to_csv(
    "../data/outputs/"
    "expected_loss_predictions.csv",
    index=False
)

In [87]:
portfolio_summary.to_csv(
    "../data/outputs/"
    "portfolio_risk_summary.csv",
    index=False
)